# 循环稳定性训练数据预检

输出：指定循环次数与工况的电容保持率。默认七项输入见模型合同。

电流使用文献报告值，质量分母未知状态保留；本笔记本不拟合模型、不显示研究预测精度。
主集8条；含无机组分敏感性集10条；再含近似数值敏感性集11条。三者嵌套，不相加。

各代码单元已通过同一 Python 命名空间依次执行并保存输出；未使用原生 Jupyter 内核执行器。
可在具备本项目依赖的 Jupyter 环境中重新执行。路径仅用于读取此版本数据库。


In [1]:
from pathlib import Path
import sys, json
ROOT = Path('D:\\机器学习-超级电容器\\循环稳定性数据库_v0.3_数据库优化_20261017\\第四轮_专业复核与训练数据_20261017')
sys.path.insert(0, str(ROOT/'modeling'))
from prepare_dataset import load_ready, preflight, CONTRACT
print(json.dumps(preflight(), ensure_ascii=False, indent=2))


{
  "status": "data_ready_for_exploratory_reported_protocol_regression",
  "default_profile": "literature_core7_exact",
  "profiles_nested_not_additive": {
    "literature_core7_exact": {
      "endpoints": 8,
      "papers": 8,
      "materials": 8,
      "tests": 8,
      "source_groups": 8,
      "unknown_current_denominator": 8,
      "approximate_labels": 0,
      "missing_optional_pore": 2
    },
    "literature_residue_sensitivity": {
      "endpoints": 10,
      "papers": 9,
      "materials": 9,
      "tests": 9,
      "source_groups": 9,
      "unknown_current_denominator": 10,
      "approximate_labels": 0,
      "missing_optional_pore": 2
    },
    "literature_approx_sensitivity": {
      "endpoints": 11,
      "papers": 10,
      "materials": 10,
      "tests": 10,
      "source_groups": 10,
      "unknown_current_denominator": 11,
      "approximate_labels": 1,
      "missing_optional_pore": 3
    }
  },
  "canonical_current_ready": 0,
  "dependencies": {
    "numpy": tr

In [2]:
rows = load_ready()
for r in rows:
    print(r['record_id'], r['cycle_count'], r['retention_pct'], r['reported_current_A_g'], r['electrolyte'], r['concentration_value'], r['voltage_lower_V'], r['voltage_upper_V'], r['BET_m2_g'])
print('输入列:', CONTRACT['required_conditions'] + ['BET_m2_g'])
print('目标列: retention_pct; 不能作为输入')


R091_CY01 5000 73.0 3.0 Li2SO4 0.5 0.0 1.8 1098.07
R111_CY01 2000 89.44 1.0 KOH 6.0 0.0 1.0 2768.0
SRC035_CY01 10000 100.0 5.0 Na2SO4 1.0 0.0 2.0 938.6
R107_CY01 10000 95.5 5.0 KOH 6.0 0.0 1.0 3006.0
R104_CY01 20000 98.4 10.0 CH3COONa 1.0 0.0 1.5 1198.0
SRC124_CY01 50000 92.0 10.0 KOH 6.0 0.0 1.0 501.0
R093_CY01 10000 108.8 5.0 KOH 6.0 0.0 1.2 2035.6
R103_CY01 10000 96.1 20.0 KOH 6.0 0.0 1.0 2285.0
输入列: ['cycle_count', 'reported_current_A_g', 'electrolyte', 'concentration_value', 'voltage_lower_V', 'voltage_upper_V', 'BET_m2_g']
目标列: retention_pct; 不能作为输入


In [3]:
plans = json.loads((ROOT/'modeling/分组验证计划_未拟合.json').read_text(encoding='utf-8'))
for profile, folds in plans.items():
    for fold in folds:
        assert not set(fold['train_groups']) & set(fold['test_groups'])
        assert len(fold['test_groups']) == 1
    print(profile, '留一来源折数:', len(folds), '研究拟合次数:0')
assert all(r['canonical_current_A_g'] is None for r in rows)
print('未知质量分母未被伪填；无目标插补；本笔记本不调用 fit。')


literature_core7_exact 留一来源折数: 8 研究拟合次数:0
literature_residue_sensitivity 留一来源折数: 9 研究拟合次数:0
literature_approx_sensitivity 留一来源折数: 10 研究拟合次数:0
未知质量分母未被伪填；无目标插补；本笔记本不调用 fit。


## 如何解读

八个来源可以检查数据接口并开展探索性验证，但循环次数与材料/来源仍然纠缠，不能宣称已经学到完整退化轨迹。
稀有电解液在留出来源时可能成为未见类别；编码器能处理并不等于模型能可靠外推。
质量分母未知、温度与负载缺失都影响解释；按来源分组不能消除这些因素。

研究训练保持关闭。合成数据接口结果位于独立的 SYNTHETIC_INTERFACE_CHECK_NOT_RESEARCH 目录，不能作为论文性能。
